In [1]:
import torch
from nnicf import denormalize
import os
import time

In [2]:
og_pt_dir = "./pt_dir/"
pt_dir = "./pt_dir_globalnorm/"
os.makedirs(og_pt_dir, exist_ok=True)
os.makedirs(pt_dir, exist_ok=True)
mod = "16qam" # Make sure this matches the data you want to fix!
# mod = "qpsk" # Make sure this matches the data you want to fix!
train_size = 80

In [4]:
print("--- STEP 1: Finding Global Limits from Training Data (Files 0 to train_size - 1) ---")
# Initialize global trackers
g_X_r_min, g_X_r_max = float('inf'), float('-inf')
g_Y_r_min, g_Y_r_max = float('inf'), float('-inf')
g_X_i_min, g_X_i_max = float('inf'), float('-inf')
g_Y_i_min, g_Y_i_max = float('inf'), float('-inf')

--- STEP 1: Finding Global Limits from Training Data (Files 0 to train_size - 1) ---


In [5]:
# Only scan the first train_size files to prevent Data Leakage!
for i in range(train_size):
    # Load dictionaries
    data_real = torch.load(os.path.join(og_pt_dir, f"{mod}_tx_rx_32_part_{i:02d}_real.pt"), weights_only=True)
    data_imag = torch.load(os.path.join(og_pt_dir, f"{mod}_tx_rx_32_part_{i:02d}_imag.pt"), weights_only=True)

    # Extract local limits and update globals
    g_X_r_min = min(g_X_r_min, data_real['X_min'])
    g_X_r_max = max(g_X_r_max, data_real['X_max'])
    g_Y_r_min = min(g_Y_r_min, data_real['Y_min'])
    g_Y_r_max = max(g_Y_r_max, data_real['Y_max'])

    g_X_i_min = min(g_X_i_min, data_imag['X_min'])
    g_X_i_max = max(g_X_i_max, data_imag['X_max'])
    g_Y_i_min = min(g_Y_i_min, data_imag['Y_min'])
    g_Y_i_max = max(g_Y_i_max, data_imag['Y_max'])
    print(f"Found Local Real X Limits: Min = {data_real['X_min']:.4f}, Max = {data_real['X_max']:.4f}")

print(f"Found Global Real X Limits: Min = {g_X_r_min:.4f}, Max = {g_X_r_max:.4f}")

Found Local Real X Limits: Min = -0.2148, Max = 0.2252
Found Local Real X Limits: Min = -0.2459, Max = 0.2211
Found Local Real X Limits: Min = -0.2358, Max = 0.2255
Found Local Real X Limits: Min = -0.2280, Max = 0.2243
Found Local Real X Limits: Min = -0.2198, Max = 0.2600
Found Local Real X Limits: Min = -0.2400, Max = 0.2280
Found Local Real X Limits: Min = -0.2380, Max = 0.2345
Found Local Real X Limits: Min = -0.2287, Max = 0.2307
Found Local Real X Limits: Min = -0.2322, Max = 0.2280
Found Local Real X Limits: Min = -0.2276, Max = 0.2296
Found Local Real X Limits: Min = -0.2294, Max = 0.2384
Found Local Real X Limits: Min = -0.2218, Max = 0.2252
Found Local Real X Limits: Min = -0.2629, Max = 0.2134
Found Local Real X Limits: Min = -0.2353, Max = 0.2461
Found Local Real X Limits: Min = -0.2241, Max = 0.2201
Found Local Real X Limits: Min = -0.2114, Max = 0.2173
Found Local Real X Limits: Min = -0.2290, Max = 0.2228
Found Local Real X Limits: Min = -0.2361, Max = 0.2305
Found Loca

In [10]:
print("\n--- STEP 2: Denormalizing and Renormalizing ALL 100 Files ---")
before_loop = time.time()
for i in range(100):
    loop_start = time.time() if i else before_loop

    real_path_og = os.path.join(og_pt_dir, f"{mod}_tx_rx_32_part_{i:02d}_real.pt")
    imag_path_og = os.path.join(og_pt_dir, f"{mod}_tx_rx_32_part_{i:02d}_imag.pt")

    data_real = torch.load(real_path_og, weights_only=True)
    data_imag = torch.load(imag_path_og, weights_only=True)

    # --- PROCESS REAL ---
    # 1. Denormalize back to raw physical voltages
    raw_X_r = denormalize(data_real['X_norm'], (data_real['X_min'], data_real['X_max']))
    raw_Y_r = denormalize(data_real['Y_norm'], (data_real['Y_min'], data_real['Y_max']))

    # todo: maybe i could make a function for that
    # 2. Renormalize using Global Limits
    data_real['X_norm'] = 2.0 * ((raw_X_r - g_X_r_min) / (g_X_r_max - g_X_r_min)) - 1.0 if g_X_r_max != g_X_r_min else raw_X_r
    data_real['Y_norm'] = 2.0 * ((raw_Y_r - g_Y_r_min) / (g_Y_r_max - g_Y_r_min)) - 1.0 if g_Y_r_max != g_Y_r_min else raw_Y_r

    # 3. OVERWRITE the old local limits with the global limits in the dictionary!
    data_real['X_min'], data_real['X_max'] = g_X_r_min, g_X_r_max
    data_real['Y_min'], data_real['Y_max'] = g_Y_r_min, g_Y_r_max

    # --- PROCESS IMAGINARY ---
    raw_X_i = denormalize(data_imag['X_norm'], (data_imag['X_min'], data_imag['X_max']))
    raw_Y_i = denormalize(data_imag['Y_norm'], (data_imag['Y_min'], data_imag['Y_max']))

    # todo: maybe i could make a function for that
    data_imag['X_norm'] = 2.0 * ((raw_X_i - g_X_i_min) / (g_X_i_max - g_X_i_min)) - 1.0 if g_X_i_max != g_X_i_min else raw_X_i
    data_imag['Y_norm'] = 2.0 * ((raw_Y_i - g_Y_i_min) / (g_Y_i_max - g_Y_i_min)) - 1.0 if g_Y_i_max != g_Y_i_min else raw_Y_i

    data_imag['X_min'], data_imag['X_max'] = g_X_i_min, g_X_i_max
    data_imag['Y_min'], data_imag['Y_max'] = g_Y_i_min, g_Y_i_max

    real_path_out = os.path.join(pt_dir, f"{mod}_tx_rx_32_part_{i:02d}_real.pt")
    imag_path_out = os.path.join(pt_dir, f"{mod}_tx_rx_32_part_{i:02d}_imag.pt")

    # Save files back to disk (overwriting)
    torch.save(data_real, real_path_out)
    torch.save(data_imag, imag_path_out)

    loop_end = time.time()
    print(f"Iteration {i + 1}/100 completed in {loop_end - loop_start:.2f} seconds")
    print(f"{loop_end - before_loop:.2f} seconds passed since before loop start")

print("Data successfully updated with Global Normalization!")


--- STEP 2: Denormalizing and Renormalizing ALL 100 Files ---
Iteration 1/100 completed in 1.24 seconds
1.24 seconds passed since before loop start
Iteration 2/100 completed in 1.76 seconds
2.99 seconds passed since before loop start
Iteration 3/100 completed in 1.29 seconds
4.28 seconds passed since before loop start
Iteration 4/100 completed in 1.25 seconds
5.53 seconds passed since before loop start
Iteration 5/100 completed in 1.13 seconds
6.66 seconds passed since before loop start
Iteration 6/100 completed in 1.29 seconds
7.95 seconds passed since before loop start
Iteration 7/100 completed in 1.70 seconds
9.65 seconds passed since before loop start
Iteration 8/100 completed in 1.21 seconds
10.86 seconds passed since before loop start
Iteration 9/100 completed in 1.65 seconds
12.51 seconds passed since before loop start
Iteration 10/100 completed in 1.06 seconds
13.57 seconds passed since before loop start
Iteration 11/100 completed in 1.31 seconds
14.89 seconds passed since bef

In [13]:
for i in range(train_size):
    # Load dictionaries
    data_real = torch.load(os.path.join(pt_dir, f"{mod}_tx_rx_32_part_{i:02d}_real.pt"), weights_only=True)
    data_imag = torch.load(os.path.join(pt_dir, f"{mod}_tx_rx_32_part_{i:02d}_imag.pt"), weights_only=True)

    # Extract local limits and update globals
    g_X_r_min = min(g_X_r_min, data_real['X_min'])
    g_X_r_max = max(g_X_r_max, data_real['X_max'])
    g_Y_r_min = min(g_Y_r_min, data_real['Y_min'])
    g_Y_r_max = max(g_Y_r_max, data_real['Y_max'])

    g_X_i_min = min(g_X_i_min, data_imag['X_min'])
    g_X_i_max = max(g_X_i_max, data_imag['X_max'])
    g_Y_i_min = min(g_Y_i_min, data_imag['Y_min'])
    g_Y_i_max = max(g_Y_i_max, data_imag['Y_max'])
    print(f"Found Local Real X Limits: Min = {g_X_r_min:.4f}, Max = {g_X_r_max:.4f}")

print(f"Found Global Real X Limits: Min = {g_X_r_min:.4f}, Max = {g_X_r_max:.4f}")

Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Local Real X Limits: Min = -0.2629, Max = 0.2600
Found Loca